In [1]:
import pandas as pd

df_test = pd.read_csv("/Users/nataliesgarcia/Desktop/dtsc4301/data/AirData_2023/August23.csv", low_memory=False)
print(df_test.columns.tolist())

['YEAR', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM', 'ORIGIN_AIRPORT_ID', 'ORIGIN_CITY_MARKET_ID', 'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR', 'DEST_AIRPORT_ID', 'DEST_CITY_MARKET_ID', 'DEST', 'DEST_CITY_NAME', 'DEST_STATE_ABR', 'DEP_TIME', 'DEP_DELAY', 'DEP_DELAY_NEW', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON', 'TAXI_IN', 'ARR_TIME', 'ARR_DELAY', 'ARR_DELAY_NEW', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'FLIGHTS', 'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY', 'TOTAL_ADD_GTIME']


In [2]:
import pandas as pd
from glob import glob

# PARAMETERS (2023 DATA)

DATA_PATH = "/Users/nataliesgarcia/Desktop/dtsc4301/data/AirData_2023/*.csv"
OUTPUT_PATH = "/Users/nataliesgarcia/Desktop/dtsc4301/data/Delayed_Cancelled_Flights_2023.csv"
DELAY_THRESHOLD = 15


# CLEANING FUNCTION (PER MONTH)


def clean_monthly_file(filepath):
    df = pd.read_csv(filepath, low_memory=False)

    # Create delay label (DOT standard)
    df["DELAYED"] = df["ARR_DELAY"] >= DELAY_THRESHOLD

    # Create disruption flag
    df["DISRUPTED"] = (df["DELAYED"]) | (df["CANCELLED"] == 1)

    # Keep only delayed or cancelled flights
    df = df[df["DISRUPTED"]]

    # Keep relevant columns (including OP_UNIQUE_CARRIER)
    df = df[
        [
            "FL_DATE",
            "MONTH",
            "DAY_OF_WEEK",
            "OP_UNIQUE_CARRIER",
            "OP_CARRIER_FL_NUM",
            "ORIGIN",
            "DEST",
            "DEP_DELAY",
            "ARR_DELAY",
            "CANCELLED",
            "CANCELLATION_CODE",
            "WEATHER_DELAY",
            "NAS_DELAY",
            "CARRIER_DELAY",
            "SECURITY_DELAY",
            "LATE_AIRCRAFT_DELAY",
            "DELAYED"
        ]
    ]

    return df


# LOAD + CLEAN + MERGE ALL 2023 MONTHS

files = glob(DATA_PATH)
print("2023 Monthly files found:", len(files))

if len(files) == 0:
    raise FileNotFoundError("No 2023 CSV files found. Check AirData_2023 path.")

monthly_dfs = []

for f in files:
    df_clean = clean_monthly_file(f)

    if not df_clean.empty:
        monthly_dfs.append(df_clean)

if len(monthly_dfs) == 0:
    raise ValueError("No delayed or cancelled flights found for 2023.")

# Combine all 2023 months
full_df_2023 = pd.concat(monthly_dfs, ignore_index=True)

print("Total 2023 delayed or cancelled flights:", full_df_2023.shape[0])

# SAVE COMBINED 2023 CSV
full_df_2023.to_csv(OUTPUT_PATH, index=False)

print("Saved 2023 cleaned dataset to:")
print(OUTPUT_PATH)

2023 Monthly files found: 12
Total 2023 delayed or cancelled flights: 1474642
Saved 2023 cleaned dataset to:
/Users/nataliesgarcia/Desktop/dtsc4301/data/Delayed_Cancelled_Flights_2023.csv


In [3]:
# Cleaned dataset
df = pd.read_csv("/Users/nataliesgarcia/Desktop/dtsc4301/data/Delayed_Cancelled_Flights_2023.csv")

print("Full dataset size:", df.shape)

# Create 10k sample
df_sample = df.sample(n=10000, random_state=42)

# Save sample
df_sample.to_csv("/Users/nataliesgarcia/Desktop/dtsc4301/data/Delayed_Cancelled_Flights_2023sample.csv", index=False)

print("Sample dataset saved.")
print("Sample size:", df_sample.shape)


Full dataset size: (1474642, 17)
Sample dataset saved.
Sample size: (10000, 17)


In [7]:
# Load your combined flight dataset
full_df = pd.read_csv("/Users/nataliesgarcia/Desktop/dtsc4301/data/Delayed_Cancelled_Flights_sample.csv")

import pandas as pd

# Read file as plain text (no CSV parsing)
with open("/Users/nataliesgarcia/Desktop/dtsc4301/data/Carrier_lookup.csv", "r", encoding="utf-8") as f:
    lines = f.readlines()

# Remove header
lines = lines[1:]

# Split ONLY on first comma
data = [line.strip().split(",", 1) for line in lines]

carrier_lookup = pd.DataFrame(data, columns=["CARRIER", "CARRIERNAME"])

# Create dictionary
carrier_map = dict(zip(carrier_lookup["CARRIER"], carrier_lookup["CARRIERNAME"]))

# Map to main dataframe
full_df["AIRLINE_NAME"] = full_df["OP_UNIQUE_CARRIER"].map(carrier_map)

print("Carrier mapping successful.")

KeyError: 'OP_UNIQUE_CARRIER'

In [ ]:
full_df = pd.read_csv("/Users/nataliesgarcia/Desktop/dtsc4301/data/Delayed_Cancelled_Flights_2023sample.csv")

import pandas as pd

# Read file as plain text (no CSV parsing)
with open("/Users/nataliesgarcia/Desktop/dtsc4301/data/Carrier_lookup.csv", "r", encoding="utf-8") as f:
    lines = f.readlines()

# Remove header
lines = lines[1:]

# Split ONLY on first comma
data = [line.strip().split(",", 1) for line in lines]

carrier_lookup = pd.DataFrame(data, columns=["CARRIER", "CARRIERNAME"])

# Create dictionary
carrier_map = dict(zip(carrier_lookup["CARRIER"], carrier_lookup["CARRIERNAME"]))

# Map to main dataframe
full_df["AIRLINE_NAME"] = full_df["OP_UNIQUE_CARRIER"].map(carrier_map)

print("Carrier mapping successful.")